# evalkit walkthrough

This notebook walks through the complete evalkit workflow using a mock model - no API keys required.

**What we cover:**
1. Planning - sample size before labelling any data
2. Dataset setup and template validation
3. Running an evaluation with bootstrap CIs
4. Comparing two models correctly (McNemar's test)
5. Error analysis - what is the model getting wrong?
6. Multiple testing correction for prompt variants
7. LLM judge validation with inter-rater agreement
8. The RigorChecker - automated statistical audit


In [1]:
# Install evalkit (skip if already installed)
# !pip install evalkit-research
#
# For all optional dependencies:
# !pip install 'evalkit-research[all]'
#
# For this notebook you need at least:
# !pip install evalkit-research numpy


## 1. Planning: how many examples do you need?

Run this before labelling any data or spending API budget. The most common mistake in LLM evaluation is starting with n=50 because that's how many examples you already have, then reporting accuracy to two decimal places as if n=50 supports that precision.

In [2]:
from evalkit import PowerAnalysis

pa = PowerAnalysis(alpha=0.05, power=0.80)

# How many examples to report accuracy to ±5%?
ci_result = pa.for_ci_precision(desired_half_width=0.05, expected_accuracy=0.75)
print(ci_result)

# How many to detect a 5 percentage-point accuracy difference between two models?
cmp_result = pa.for_proportion_difference(effect_size=0.05, p1=0.75)
print(cmp_result)

Power Analysis (CI_Precision)
  Effect size: 0.0500
  α: 0.050, target power: 0.80
  Minimum N required: 289
Power Analysis (TwoProportionZ)
  Effect size: 0.0500
  α: 0.050, target power: 0.80
  Minimum N required: 1094


In [3]:
# Full planning table across effect sizes and power levels
# Screenshot this and put it in your design doc
pa.sample_size_table(
    effect_sizes=[0.02, 0.05, 0.10, 0.15, 0.20],
    powers=[0.70, 0.80, 0.90],
)

Sample Size Requirements  (α=0.05, two-tailed)
Test: TwoProportionZ
Baseline accuracy: 0.70

 Effect size │  Power 70%   Power 80%   Power 90%
─────────────┼────────────────────────────────────
      Δ = 2% │    6,354     8,080    10,816
      Δ = 5% │      984     1,251     1,674
     Δ = 10% │      231       294       392
     Δ = 15% │       96       121       161
     Δ = 20% │       49        62        82


'Sample Size Requirements  (α=0.05, two-tailed)\nTest: TwoProportionZ\nBaseline accuracy: 0.70\n\n Effect size │  Power 70%   Power 80%   Power 90%\n─────────────┼────────────────────────────────────\n      Δ = 2% │    6,354     8,080    10,816\n      Δ = 5% │      984     1,251     1,674\n     Δ = 10% │      231       294       392\n     Δ = 15% │       96       121       161\n     Δ = 20% │       49        62        82'

## 2. Dataset setup

We will use n=400 examples - adequately powered to report accuracy to ±5% and to detect a 10-point accuracy difference.

In a real evaluation, you would load from JSONL: `EvalDataset.from_jsonl("data.jsonl")`

In [4]:
from evalkit import EvalDataset, PromptTemplate

# Build a dataset in-memory for this walkthrough
qa_pairs = [
    ("What is the capital of France?", "Paris"),
    ("What is 12 × 12?", "144"),
    ("Who wrote Pride and Prejudice?", "Jane Austen"),
    ("What is the chemical symbol for gold?", "Au"),
    ("How many sides does a hexagon have?", "6"),
    ("What year did the Berlin Wall fall?", "1989"),
    ("What planet is closest to the Sun?", "Mercury"),
    ("What is the square root of 256?", "16"),
] * 50  # 400 examples total

records = [{"id": str(i), "question": q, "label": a} for i, (q, a) in enumerate(qa_pairs)]
dataset = EvalDataset.from_list(records, name="qa_benchmark")

print(f"Dataset: {len(dataset)} examples")
print(f"Label distribution: {dict(list(dataset.label_distribution().items())[:3])} ...")

Dataset: 400 examples
Label distribution: {'Paris': 50, '144': 50, 'Jane Austen': 50} ...


In [5]:
# Validate your template before spending any API budget
# This catches field-name mismatches at zero cost
template = PromptTemplate("Answer concisely: {{ question }}")
errors = template.validate(dataset)

if errors:
    print("TEMPLATE ERRORS:\n" + "\n".join(errors))
else:
    print("✓ Template is compatible with dataset")

# Try a bad template to see what errors look like
bad = PromptTemplate("{{ question }} - context: {{ context }}")
bad_errors = bad.validate(dataset)
print(f"\nBad template errors: {bad_errors[0]}")

✓ Template is compatible with dataset

Bad template errors: Example id='0': Template rendering failed. Fields provided: ['question']. Error: 'context' is undefined


## 3. Running an evaluation with bootstrap CIs

The `Experiment` class runs the pre-flight audit, evaluation, metric computation, and post-hoc audit in sequence.

In [6]:
from evalkit import ExactMatchJudge, Experiment, MockRunner

judge = ExactMatchJudge()
runner_a = MockRunner(judge=judge, template=template, accuracy=0.82, seed=42)

result_a = Experiment(
    "gpt-4o",
    dataset,
    runner_a,
    n_resamples=5_000,
).run()

result_a.print_summary()


Experiment: gpt-4o
Dataset: qa_benchmark (n=400)
Model:   mock-model-v1
Cost:    $0.0000 | Tokens: 0 | Time: 0.0s

Metrics (with 95% bootstrap CIs):
  Accuracy: 0.8425 (95% CI: 0.8075–0.8775, n=400)

╔══════════════════════════════════════════════════════╗
║           evalkit  RigorChecker  Report              ║
╚══════════════════════════════════════════════════════╝
Experiment: gpt-4o
Status: PASS  (0 errors, 1 warnings)

🟡 [UNDERPOWERED_COMPARISON] Your sample size (n=400) achieves only 55% power to detect a 5% accuracy difference. Target is 80%. A negative result here is inconclusive, not evidence of equivalence.
   → To achieve 80% power, you need n≥721. If models appear similar in accuracy, increase N before concluding equivalence.



In [7]:
# The CI tells you the actual precision of your measurement
acc = result_a.metrics["Accuracy"]
print(f"Point estimate: {acc.value:.4f}")
print(f"95% CI: [{acc.ci_lower:.4f}, {acc.ci_upper:.4f}]")
print(f"CI half-width (margin of error): ±{acc.margin_of_error:.4f}")
print()
print(f"Reporting 'accuracy = {acc.value:.2f}' implies ±0.005 precision.")
print(f"Your actual precision is ±{acc.margin_of_error:.3f}.")

Point estimate: 0.8425
95% CI: [0.8075, 0.8775]
CI half-width (margin of error): ±0.0350

Reporting 'accuracy = 0.84' implies ±0.005 precision.
Your actual precision is ±0.035.


## 4. Comparing two models correctly

In [8]:
runner_b = MockRunner(judge=judge, template=template, accuracy=0.72, seed=99)
result_b = Experiment("model_b", dataset, runner_b, n_resamples=5_000).run()

# compare() auto-selects the correct test, verifies alignment, gives actionable output
comparison = result_a.compare(result_b)
print(comparison)

Comparison: gpt-4o vs model_b
  gpt-4o: accuracy = 0.8425
  model_b: accuracy = 0.7150
  McNemar: stat=17.4825, p=0.0000, effect=2.0968 → REJECT H₀ (α=0.05)
  ✓ gpt-4o is statistically better (p=0.0000)


In [9]:
# Low-level McNemar if you want the full test result
from evalkit import McNemarTest

test = McNemarTest(alpha=0.05).test(result_a.run_result.correct, result_b.run_result.correct)
print(test)
print(f"\nEffect size (odds ratio): {test.effect_size:.2f}")
print(
    f"Interpretation: model_a is {test.effect_size:.1f}x more likely"
    " to be correct on discordant pairs"
)

McNemar: stat=17.4825, p=0.0000, effect=2.0968, n=400 → REJECT H₀ (α=0.05)

Effect size (odds ratio): 2.10
Interpretation: model_a is 2.1x more likely to be correct on discordant pairs


## 5. Error analysis

Aggregate accuracy tells you *how often* the model is wrong. `worst_examples()` tells you *how* it's wrong.

In [10]:
# The examples the model most confidently got wrong
worst = result_a.worst_examples(5)
for ex in worst:
    print(f"Q: {ex['prompt'][:60]}")
    print(f"   Output:    {ex['output']!r}")
    print(f"   Reference: {ex['reference']!r}")
    print()

Q: Answer concisely: What is the capital of France?
   Output:    '__wrong_35__'
   Reference: 'Paris'

Q: Answer concisely: What year did the Berlin Wall fall?
   Output:    '__wrong_48__'
   Reference: '1989'

Q: Answer concisely: What planet is closest to the Sun?
   Output:    '__wrong_33__'
   Reference: 'Mercury'

Q: Answer concisely: What is the capital of France?
   Output:    '__wrong_48__'
   Reference: 'Paris'

Q: Answer concisely: What is the chemical symbol for gold?
   Output:    '__wrong_57__'
   Reference: 'Au'



In [11]:
# Full results as a DataFrame - filter, sort, group by any column
df = result_a.to_dataframe()
print(f"Shape: {df.shape}")
print(f"Accuracy from DataFrame: {df['is_correct'].mean():.4f}")
print()

# Filter to incorrect answers
wrong = df[~df["is_correct"]]
print(f"Wrong answers: {len(wrong)}")
wrong[["example_id", "output", "reference"]].head()

Shape: (400, 8)
Accuracy from DataFrame: 0.8425

Wrong answers: 63


,example_id,output,reference
0,0,__wrong_35__,Paris
5,5,__wrong_48__,1989
6,6,__wrong_33__,Mercury
8,8,__wrong_48__,Paris
11,11,__wrong_57__,Au


## 6. Multiple testing correction

Running 20 prompt variants? You expect one false positive by chance at α=0.05. Benjamini-Hochberg controls the expected false discovery rate.

In [12]:
from evalkit import BHCorrection

# Simulate 6 prompt variants: Prompt A is genuinely better, B looks significant but isn't
p_values = [0.003, 0.041, 0.068, 0.24, 0.51, 0.78]
names = [f"Prompt {chr(65 + i)}" for i in range(6)]

bh_result = BHCorrection(alpha=0.05).correct(p_values, comparison_names=names)
print(bh_result)

if bh_result.false_positive_warning:
    print("\n⚠ Without FDR correction, you would ship a prompt that isn't actually better.")

1 comparison(s) appear significant without FDR correction but not after. These are likely false positives. Always report BH-adjusted p-values.


Benjamini-Hochberg FDR Correction
  Expected false positives (α=0.05): 0.30
  ⚠ Unadjusted p-values would lead to different conclusions. Always report adjusted values.
  ✓ Prompt A: p_raw=0.0030 → p_adj=0.0180
  ✗ Prompt B: p_raw=0.0410 → p_adj=0.1230
  ✗ Prompt C: p_raw=0.0680 → p_adj=0.1360
  ✗ Prompt D: p_raw=0.2400 → p_adj=0.3600
  ✗ Prompt E: p_raw=0.5100 → p_adj=0.6120
  ✗ Prompt F: p_raw=0.7800 → p_adj=0.7800

⚠ Without FDR correction, you would ship a prompt that isn't actually better.


## 7. LLM judge validation

Before trusting LLM-as-judge scores, measure inter-rater agreement. A κ < 0.60 means the judge is inconsistent enough that scores are unreliable.

In [13]:
import numpy as np

from evalkit import CohenKappa

rng = np.random.default_rng(7)
true_labels = rng.choice([0, 1], size=200, p=[0.4, 0.6]).tolist()

# Good judge: 88% agreement with reference
good_judge = [lab if rng.random() > 0.12 else 1 - lab for lab in true_labels]
# Bad judge: 68% agreement (vague rubric)
bad_judge = [lab if rng.random() > 0.32 else 1 - lab for lab in true_labels]

good_kappa = CohenKappa(n_resamples=3_000).compute(good_judge, true_labels)
bad_kappa = CohenKappa(n_resamples=3_000).compute(bad_judge, true_labels)

print(f"Good judge: {good_kappa}")
print(f"Bad judge:  {bad_kappa}")
print()
print("The RigorChecker will flag bad_kappa as LOW_JUDGE_AGREEMENT (ERROR level)")
print("Fix: make your rubric more specific, add examples to the judge prompt")

Cohen's kappa=0.455 (moderate) is below the acceptable threshold (0.60). Judgments from this rater pair are unreliable.


Good judge: ✓ CohenKappa: 0.7830 (95% CI: 0.6881–0.8637, n=200) [substantial]
Bad judge:  ✗ CohenKappa: 0.4550 (95% CI: 0.3252–0.5743, n=200) [moderate] - below minimum threshold (0.6)

The RigorChecker will flag bad_kappa as LOW_JUDGE_AGREEMENT (ERROR level)
Fix: make your rubric more specific, add examples to the judge prompt


## 8. The RigorChecker - what a bad experiment looks like

The RigorChecker runs automatically inside `Experiment.run()`. You can also call it directly.

In [14]:
from evalkit import RigorChecker

checker = RigorChecker()

# A real experiment with all the common problems baked in
bad_report = checker.audit(
    n_examples=47,  # underpowered
    accuracy=0.68,
    label_distribution={"correct": 43, "incorrect": 4},  # 91% class imbalance
    n_variants=8,  # 8 prompt variants tested
    p_values=[0.03, 0.04, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70],  # uncorrected
    judge_kappa=0.41,  # bad judge agreement
    experiment_name="typical_llm_paper_eval",
)

print(bad_report)

2 comparison(s) appear significant without FDR correction but not after. These are likely false positives. Always report BH-adjusted p-values.


╔══════════════════════════════════════════════════════╗
║           evalkit  RigorChecker  Report              ║
╚══════════════════════════════════════════════════════╝
Experiment: typical_llm_paper_eval
Status: FAIL  (3 errors, 1 warnings)

🔴 [SEVERE_CLASS_IMBALANCE] Your test set is 91% class 'correct'. A trivial model predicting the majority class achieves 91% accuracy. Accuracy is a meaningless metric here.
   → Report balanced accuracy, macro-F1, or AUC instead of accuracy. Consider stratified sampling to balance the test set.

🔴 [MULTIPLE_TESTING_UNCORRECTED] 2 comparison(s) appear significant without FDR correction but are NOT significant after Benjamini-Hochberg correction. Reporting uncorrected results would be misleading.
   → Report only BH-adjusted p-values. The apparent significant results are likely false positives.

🔴 [LOW_JUDGE_AGREEMENT] LLM-as-judge agreement (κ=0.41) is below the minimum threshold (κ=0.60). Your evaluation scores are unreliable.
   → Improve your j

In [15]:
# The same experiment done right
good_report = checker.audit(
    n_examples=400,
    accuracy=0.82,
    label_distribution=dataset.label_distribution(),
    n_variants=1,
    experiment_name="model_a_evaluation",
)

print(good_report)

╔══════════════════════════════════════════════════════╗
║           evalkit  RigorChecker  Report              ║
╚══════════════════════════════════════════════════════╝
Experiment: model_a_evaluation
Status: PASS  (0 errors, 1 warnings)

🟡 [UNDERPOWERED_COMPARISON] Your sample size (n=400) achieves only 50% power to detect a 5% accuracy difference. Target is 80%. A negative result here is inconclusive, not evidence of equivalence.
   → To achieve 80% power, you need n≥822. If models appear similar in accuracy, increase N before concluding equivalence.



## Next steps

- **Real model**: replace `MockRunner` with `AsyncRunner(provider=OpenAIProvider("gpt-4o-mini"), ...)`
- **Real dataset**: `EvalDataset.from_jsonl("your_data.jsonl")` or `.from_huggingface("squad", split="validation")`
- **HTML report**: `ReportGenerator().generate(result, output_path="report.html")`
- **CLI**: `evalkit run data.jsonl --model gpt-4o-mini --output report.html`
- **Docker**: `docker compose up api` for the REST API

See the [README](../README.md) for full documentation.
